# Projekt zaliczeniowy: Zastosowanie sieci neuronowych

Szkielet notebooka projektu. Wymagania, tematy i kryteria oceny są w pliku `projekt_opis.md`. Sekcje odpowiadają strukturze z opisu. Każdą sekcję uzupełnijcie kodem i komentarzem: sam kod i wykresy bez wyjaśnienia nie wystarczą.

Środowisko: Google Colab z GPU (Środowisko wykonawcze, Zmień typ środowiska wykonawczego, T4 GPU). Notebook musi przechodzić od początku do końca po wybraniu "Uruchom wszystko".

Jeżeli w projekcie dostrajacie model z biblioteki transformers (na przykład HerBERT), notebook musi działać na backendzie torch, tak jak w laboratorium 6: ustawcie `KERAS_BACKEND` w pierwszej komórce kodu, przed importem `keras`, i uruchamiajcie go w świeżym środowisku wykonawczym.

## 0. Konfiguracja

Instalacja bibliotek spoza Colaba, ziarno losowania i wersje bibliotek. Parametry, które zmieniacie między wersją pełną a skróconą (liczba epok, podzbiór danych), trzymajcie w jednym miejscu, w komórce `KONFIGURACJA`.

In [ ]:
import os, sys
# Dla modeli z transformers (jak w lab 6) odkomentujcie, zanim zaimportujecie keras:
# os.environ["KERAS_BACKEND"] = "torch"

if "google.colab" in sys.modules:
    %pip install -q keras-tuner

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)

print("Python", sys.version.split()[0])
print("Keras", keras.__version__, "backend", keras.backend.backend())
if keras.backend.backend() == "tensorflow":
    import tensorflow as tf
    print("TensorFlow", tf.__version__, "GPU:", tf.config.list_physical_devices("GPU"))
elif keras.backend.backend() == "torch":
    import torch
    print("torch", torch.__version__, "CUDA:", torch.cuda.is_available())

In [ ]:
KONFIGURACJA = {
    "epoki": 20,          # TODO: wartości dla pełnego przebiegu
    "batch_size": 32,
    "podzbior": None,     # np. 2000 dla szybkiej wersji kontrolnej, None = całe dane
}
KONFIGURACJA

### Funkcje pomocnicze

Gotowe funkcje, z których możecie korzystać. Nie musicie ich używać, jeśli macie własne.

In [ ]:
def wczytaj_obrazy_parquet(url, rozmiar=(160, 160), kolumna_obrazu="image", kolumna_etykiety="label", limit=None):
    """Wczytuje plik Parquet z obrazami w formacie Hugging Face (słownik z bajtami JPEG lub PNG).
    Zwraca tablicę uint8 o kształcie (n, wys, szer, 3) i wektor etykiet."""
    import io
    from PIL import Image
    df = pd.read_parquet(url, columns=[kolumna_obrazu, kolumna_etykiety])
    if limit is not None:
        df = df.iloc[:limit]
    X = np.empty((len(df), rozmiar[0], rozmiar[1], 3), dtype=np.uint8)
    for i, rekord in enumerate(df[kolumna_obrazu]):
        img = Image.open(io.BytesIO(rekord["bytes"])).convert("RGB").resize((rozmiar[1], rozmiar[0]))
        X[i] = np.asarray(img)
    y = df[kolumna_etykiety].to_numpy()
    del df
    return X, y


def rysuj_krzywe(historia, metryka="accuracy", tytul=""):
    """Rysuje funkcję straty i wybraną metrykę dla zbioru treningowego i walidacyjnego."""
    h = historia.history
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(h["loss"], label="trening"); ax[0].plot(h.get("val_loss", []), label="walidacja")
    ax[0].set_title(f"Funkcja straty {tytul}"); ax[0].set_xlabel("epoka"); ax[0].legend()
    if metryka in h:
        ax[1].plot(h[metryka], label="trening"); ax[1].plot(h.get("val_" + metryka, []), label="walidacja")
        ax[1].set_title(f"{metryka} {tytul}"); ax[1].set_xlabel("epoka"); ax[1].legend()
    plt.tight_layout(); plt.show()


def bootstrap_ci(y_true, y_pred, metryka, n=1000, alfa=0.05, seed=RANDOM_STATE):
    """Przedział ufności metryki metodą bootstrap: losujemy ze zwracaniem przykłady testowe."""
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    wyniki = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        wyniki.append(metryka(y_true[idx], y_pred[idx]))
    return metryka(y_true, y_pred), np.quantile(wyniki, alfa / 2), np.quantile(wyniki, 1 - alfa / 2)


def czas_predykcji(model, x_jeden, powtorzenia=50):
    """Mediana czasu predykcji jednego przykładu w milisekundach (po rozgrzewce).
    Używa predict_on_batch, bo model.predict ma duży stały narzut przy pojedynczym przykładzie."""
    model.predict_on_batch(x_jeden)
    czasy = []
    for _ in range(powtorzenia):
        t0 = time.perf_counter(); model.predict_on_batch(x_jeden); czasy.append(time.perf_counter() - t0)
    return 1000 * float(np.median(czasy))

## 1. Autor, problem, cel

Imię i nazwisko, wybrany temat, jedno lub dwa zdania o tym, do czego model mógłby służyć.

Odpowiedź:

## 2. Dane: źródło, licencja, opis, przykłady

Wczytajcie dane z publicznego adresu. Podajcie źródło i licencję, liczbę przykładów w każdym zbiorze, rozkład klas i kilka przykładów (obrazy z etykietami albo fragmenty tekstów).

Przykłady wczytania (adresy z `projekt_opis.md`):

```python
# obrazy
X_tr, y_tr = wczytaj_obrazy_parquet("https://huggingface.co/datasets/AI-Lab-Makerere/beans/resolve/main/data/train-00000-of-00001.parquet",
                                    rozmiar=(160, 160), kolumna_etykiety="labels")
# tekst
df_tr = pd.read_csv("https://huggingface.co/datasets/allegro/klej-cbd/resolve/main/train.csv")
```

In [ ]:
# TODO: wczytanie danych

In [ ]:
# TODO: rozkład klas i przykłady

Komentarz do danych (licencja, jakość, czego się spodziewacie):

## 3. Podział na zbiory i protokół ewaluacji

Opiszcie podział (skąd walidacja, czy podział jest stratyfikowany albo grupowany), metrykę główną i powód jej wyboru. Zbiór testowy odkładacie do sekcji 8.

In [ ]:
# TODO: podział na zbiory

Metryka główna i uzasadnienie:

## 4. Punkt odniesienia

Wynik klasy większościowej i prosty model niebędący głęboką siecią, oba na zbiorze walidacyjnym.

In [ ]:
# TODO: klasa większościowa

In [ ]:
# TODO: prosty model (np. regresja logistyczna)

## 5. Sieć trenowana od zera

Architektura zaprojektowana samodzielnie, krzywe uczenia (`rysuj_krzywe`) i komentarz o przeuczeniu lub niedouczeniu.

In [ ]:
model_od_zera = None  # TODO: definicja, kompilacja i trening

Komentarz:

## 6. Model pretrenowany

Transfer learning (obrazy) albo fine-tuning modelu typu BERT (tekst), porównany z siecią z sekcji 5 przy podobnym budżecie treningu.

In [ ]:
model_pretrenowany = None  # TODO

Komentarz:

## 7. Regularyzacja i strojenie hiperparametrów

Co najmniej jedna technika regularyzacji i systematyczne przeszukanie co najmniej dwóch hiperparametrów. Wyniki zbierzcie w tabeli (konfiguracja, wynik walidacyjny, czas).

In [ ]:
eksperymenty = []  # TODO: lista słowników {"konfiguracja": ..., "wynik_val": ..., "czas_s": ...}
pd.DataFrame(eksperymenty)

Komentarz:

## 8. Wybór modelu i jednorazowy test

Wybierzcie model na podstawie walidacji, napiszcie dlaczego, i dopiero teraz policzcie wynik na zbiorze testowym: metryka główna, `classification_report`, macierz pomyłek.

In [ ]:
# TODO: ocena na zbiorze testowym (jeden raz)

## 9. Analiza błędów

Przykłady błędnych predykcji z interpretacją. Dla obrazów Grad-CAM dla kilku błędów (laboratorium 4), dla tekstu przykłady i próba wskazania wzorca.

In [ ]:
# TODO: analiza błędów

Komentarz:

## 10. Niepewność, odporność i koszt

Wymagania na ocenę 5,0: powtórzenia z różnymi ziarnami albo `bootstrap_ci`, test na danych spoza rozkładu treningowego, tabela kosztów (parametry, rozmiar pliku, czas treningu, `czas_predykcji`).

In [ ]:
# TODO: niepewność wyniku

In [ ]:
# TODO: odporność

In [ ]:
# TODO: koszt

## 11. Wnioski, ograniczenia, notatka dla odbiorcy nietechnicznego

Wnioski:

Ograniczenia:

Notatka (po polsku, bez żargonu, do jednej strony):